# Starfysh Slide-seq tutorial in scviva-tools

This notebook adapts the upstream Starfysh Slide-seq tutorial to the current `scviva.external.Starfysh` wrapper.

The active path covers expression-only deconvolution. Upstream Starfysh preprocessing, plotting, and result serialization helpers are represented with local notebook code or future-capability notes.


In [ ]:
!pip install --quiet scviva-tools


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch

import scviva
from scviva.external import Starfysh

scviva.settings.seed = 0
torch.manual_seed(0)


In [ ]:
def _as_dense(x):
    if hasattr(x, "toarray"):
        return x.toarray()
    return np.asarray(x)


def compute_signature_scores(adata, gene_signatures):
    """Compute simple per-spot signature priors from marker-gene columns.

    The current scviva Starfysh wrapper expects one prior score per spot and
    cell type. Upstream Starfysh notebooks often start from marker-gene lists;
    this helper turns those marker lists into normalized spot-level priors.
    """
    signatures = gene_signatures.copy()
    if "Unnamed: 0" in signatures.columns:
        signatures = signatures.drop(columns=["Unnamed: 0"])

    scores = pd.DataFrame(index=adata.obs_names)
    for cell_type in signatures.columns:
        markers = signatures[cell_type].dropna().astype(str)
        markers = [gene for gene in markers if gene in adata.var_names]
        if len(markers) == 0:
            scores[cell_type] = 0.0
            continue
        values = _as_dense(adata[:, markers].layers.get("counts", adata[:, markers].X))
        scores[cell_type] = values.mean(axis=1)

    scores = scores.clip(lower=0)
    row_sums = scores.sum(axis=1).replace(0, np.nan)
    return scores.div(row_sums, axis=0).fillna(1.0 / scores.shape[1])


## Load Slide-seq counts, coordinates, and signatures

Update these paths to point at the files used by the upstream tutorial. Counts are expected as genes by spots, matching the source notebook; the cell below transposes them into spots by genes for AnnData.


In [ ]:
counts_path = Path("spatial/MPM08_on_later_counts.csv")
coords_path = Path("spatial/MPM08_on_later_coords.csv")
signature_path = Path("full_sigs.csv")

counts = pd.read_csv(counts_path, index_col=0)
coords = pd.read_csv(coords_path, index_col=0)
if not signature_path.exists():
    raise FileNotFoundError(
        f"Signature file not found at {signature_path}. "
        "Provide the full_sigs.csv marker gene table."
    )
gene_sig = pd.read_csv(signature_path, index_col=0)

adata = sc.AnnData(X=counts.T.astype(np.float32))
adata.layers["counts"] = adata.X.copy()
coords = coords.reindex(adata.obs_names)
# Verify spot barcodes in counts match those in coords
assert set(adata.obs_names).issubset(coords.index), (
    "Spot barcodes in counts CSV columns do not match coords CSV index. "
    "Check that the files come from the same Slide-seq experiment."
)
# Prefer physical coordinates (x, y); fall back to grid coordinates (array_row, array_col).
# Physical coords are in microns for Slide-seq; grid coords are integer array positions for Visium.
coord_columns = [col for col in ["x", "y", "array_row", "array_col"] if col in coords.columns]
if len(coord_columns) < 2:
    raise ValueError(
        f"Expected coordinate columns 'x','y' or 'array_row','array_col' in {coords_path}. "
        f"Found: {list(coords.columns)}"
    )
adata.obsm["spatial"] = coords[coord_columns[:2]].to_numpy(dtype=np.float32)

signature_scores = compute_signature_scores(adata, gene_sig)
signature_scores.head()


## Train expression-only Starfysh

The current scviva wrapper accepts spot-level signature scores as priors, registers counts and spatial coordinates, then returns proportions and latent/model outputs.


> **Note: raw counts required.** The Starfysh model expects raw integer count data in the registered layer (default: `adata.X`). Do **not** log-normalize or scale before calling `setup_anndata` — the model applies its own library-size normalization internally. Normalizing beforehand will silently degrade training.

In [ ]:
Starfysh.setup_anndata(adata, layer="counts", spatial_key="spatial")
model = Starfysh(
    adata,
    signature_scores=signature_scores,
    n_latent=10,
    n_hidden=128,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.train(max_epochs=100, batch_size=128, lr=1e-3, device=device, prog_bar=True)


In [ ]:
proportions = model.get_proportions(batch_size=128, store_key="starfysh_proportions")
latent = model.get_latent_representation(batch_size=128, store_key="X_starfysh")
outputs = model.get_model_outputs(batch_size=128, store=True)

print("proportions:", proportions.shape)
print("latent:", latent.shape)
print("px_rate:", outputs["px_rate"].shape)
proportions.head()


In [ ]:
for cell_type in proportions.columns[: min(4, proportions.shape[1])]:
    adata.obs[f"starfysh_{cell_type}"] = proportions[cell_type].values

sc.pl.embedding(
    adata,
    basis="spatial",
    color=[f"starfysh_{ct}" for ct in proportions.columns[: min(4, proportions.shape[1])]],
    frameon=False,
    ncols=2,
)


## Deferred upstream sections

The source Slide-seq notebook includes Starfysh-specific library smoothing, anchor detection, plotting helpers, and JSON result serialization. These are planned for later scviva phases and should remain disabled until the corresponding APIs exist.


In [ ]:
# TODO: Phase 6 — Starfysh preprocessing and plotting helpers
# Upstream reference only:
#   win_loglib = utils.get_windowed_library(...)
#   pure_spots, pure_dict, pure_idx = utils.get_anchor_spots(...)
#   plot_utils.pl_spatial_feature(...)
#   post_analysis.pred_prop_scatter(...)
